# Sàng lọc policy ứng viên

So từng ứng viên trong `sbx/candidates.py` với policy tham chiếu (public LB 0,01582), có nhóm null để biết đâu là nhiễu.

| Bước | Kịch bản | Độ dài | Mục đích |
|---|---|---|---|
| 1 · Sàng nhanh | 10 kịch bản đóng góp nhiều nhất | 7 ngày | Loại nhanh ứng viên kém |
| 2 · Xác nhận | 30 kịch bản | 30 ngày | Con số thật cho 3 ứng viên tốt nhất |

Cần: repo này + starter kit (Python 3.12). Kết quả lưu ở `results/screen/`, chạy lại sẽ bỏ qua phần đã xong.

In [ ]:
import os, sys, json, time
from pathlib import Path

# Find the repo (the folder holding sbx/) and the starter kit.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sbx").is_dir())
sys.path.insert(0, str(REPO))
# os.environ["SMART_BUILDINGS_DATA_ROOT"] = "/path/to/starter-kit"   # if not next to the repo
from sbx import paths
DATA_ROOT = paths.setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sbx import evaluation as ev, candidates as cand

WORKERS = int(os.getenv("SB_WORKERS", 0)) or os.cpu_count()   # set to your physical core count
print("repo:", REPO, "\ndata:", DATA_ROOT, "\nworkers:", WORKERS)

In [ ]:
QUICK_STEPS = 2016          # 7 days. Short runs skip some weekends; use only to rank.
FULL_STEPS = 8640
CONFIRM_TOP = 3
OUT = REPO / "results" / "screen"
NULLS = ["null_1", "null_2", "null_3"]
CANDS = cand.CANDIDATES      # or a subset, e.g. ["fan_tr", "optimal_start", "night_flush"]
print("candidates:", CANDS)

In [ ]:
def default_screen_set(phase0_csv=None, n=10):
    """Top contributors from the phase-0 run if available, else two scenarios per station."""
    for p in [phase0_csv, REPO / "notebooks" / "eval_ref" / "scores_ref.csv", REPO / "eval_ref" / "scores_ref.csv"]:
        if p and Path(p).exists():
            df = pd.read_csv(p).sort_values("score", ascending=False)
            print("screen set: top", n, "contributors from", p)
            return df.scenario.head(n).tolist()
    allsc = paths.training_scenarios()
    by_station = {}
    for s in allsc:
        by_station.setdefault(s.split("/")[1].rsplit("_", 1)[0], []).append(s)
    pick = [s for v in by_station.values() for s in v[::3][:2]]
    print("screen set: 2 per station (no phase-0 results found)")
    return pick

SCREEN = default_screen_set()
SCREEN

## Bước 1 · Sàng nhanh

In [ ]:
specs = ["ref"] + NULLS + CANDS
n_jobs = len(specs) * len(SCREEN)
print(f"{n_jobs} runs of {QUICK_STEPS} steps")
ev.run_batch([(s, sp, QUICK_STEPS) for sp in specs for s in SCREEN], OUT, workers=WORKERS)

In [ ]:
quick = ev.load_results(OUT, steps=QUICK_STEPS)
quick = quick[quick.scenario.isin(SCREEN)]

# Noise per scenario from ref + null runs.
grp = quick[quick.spec.isin(["ref"] + NULLS)]
noise = grp.groupby("scenario").score.agg(["mean", "std"])
print(f"noise: median sd {100 * (noise['std'] / noise['mean']).median():.1f}% of the score")

table = ev.compare(quick[~quick.spec.isin(NULLS)], "ref", noise=noise)
display(table.round(5))

In [ ]:
piv = quick.pivot_table(index="spec", columns="scenario", values="score")
chg = 100 * (piv - piv.loc["ref"]) / piv.loc["ref"]
chg = chg.drop(index=["ref"] + [n for n in NULLS if n in chg.index]).loc[table.index.drop("ref", errors="ignore")]
fig, ax = plt.subplots(figsize=(1.0 + 0.9 * chg.shape[1], 0.45 * chg.shape[0] + 1.5))
lim = max(5, np.nanpercentile(np.abs(chg.to_numpy()), 95))
im = ax.imshow(chg.to_numpy(), cmap="RdYlGn_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(chg.shape[1]), [c.replace("building_train_", "B").replace("/", "\n") for c in chg.columns], fontsize=8)
ax.set_yticks(range(chg.shape[0]), chg.index, fontsize=9)
for i in range(chg.shape[0]):
    for j in range(chg.shape[1]):
        v = chg.iat[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:+.0f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="% thay đổi so với ref (xanh = tốt hơn)")
ax.set_title(f"Sàng nhanh: {QUICK_STEPS} bước")
fig.tight_layout(); plt.show()

## Bước 2 · Xác nhận trên 30 kịch bản đầy đủ

In [ ]:
ok = table.drop(index="ref", errors="ignore")
ok = ok[ok.change_pct < 0]
if "z_worse" in ok:
    ok = ok[ok.z_worse == 0]           # drop anything that clearly hurts a scenario
best = ok.sort_values("score").index[:CONFIRM_TOP].tolist()
print("confirming:", best)
ALL = paths.training_scenarios()
ev.run_batch([(s, sp, FULL_STEPS) for sp in ["ref"] + best for s in ALL], OUT, workers=WORKERS)

In [ ]:
full = ev.load_results(OUT, steps=FULL_STEPS)
full = full[full.spec.isin(["ref"] + best)]
final = ev.compare(full, "ref")
display(final.round(5))
by_station = full.pivot_table(index="station", columns="spec", values="score")
display(by_station.round(4))
final.to_csv(OUT / "confirm_summary.csv")
table.to_csv(OUT / "quick_summary.csv")

**Đọc kết quả:** một ứng viên đáng giữ khi `change_pct` âm trên 30 kịch bản, không làm tệ hơn rõ rệt kịch bản nào (`worst_change_pct` nhỏ),
và cải thiện lớn hơn mức nhiễu. Ứng viên tốt có thể kết hợp với nhau (vd. `combo`) rồi sàng lại.